# 📡 Global Telecom MNO — Impact Analysis Prediction
**Rule-based engine** that predicts all 24 Impact Analysis columns for every operator across MECA, Europe, APAC, and LATAM.

### Sections
1. Load & clean data
2. Helper parsers
3. Market Growth Signals (rows 1–5)
4. ARPU & Revenue Signals (rows 6–11)
5. Financial & MS Need Signals (rows 12–16)
6. Investment & Business Model Signals (rows 17–20)
7. Roaming Signals (rows 21–22)
8. Competitive Signals (rows 23–28)
9. Export to Excel (one sheet per region + Combined)

## 1. Load & Clean Data

In [4]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# ── Load ──────────────────────────────────────────────────────────────────────
FILE = '/content/Global_Telecom_MNO_Verified.xlsx'

df = pd.read_excel(FILE, sheet_name='Combined MNO Data', header=1)
print(f'Loaded {len(df)} operators | Columns: {len(df.columns)}')
df.columns

Loaded 517 operators | Columns: 35


Index(['Region', 'Sub-Region', 'Country', 'Formerly Known As',
       'Population (mln)', 'Mobile Users (mln)', 'Mobile Penetration (%)',
       'GDP Growth (%)', 'Avg Age (Yrs)', 'Age Over 65 (%)',
       'Internet Users (%)', 'GDP per Capita (USD)', 'Operator Name',
       'Sub Base (mln)', 'Market Share (%)', 'Subscriber Growth (%)',
       'Prepaid / Postpaid', 'ARPU Growth', '5G Penetration (%)',
       'Revenue Growth (3 Yrs)', 'Profitability (3 Yrs)', 'Capex / Investment',
       'Financial Comments', 'Outbound Roaming Trend', 'Inbound Roaming Trend',
       'Business Travellers Outbound', 'Top Roaming Countries',
       'OTT / International Calls', 'Roaming Comments',
       'Regulations (Last 3 Yrs)', 'Impact of Regulations',
       'Regulation Comments', 'Key Events', 'Event Month', 'Event Impact'],
      dtype='object')

In [5]:
# ── Numeric normaliser ────────────────────────────────────────────────────────
def to_float(val, default=np.nan):
    """Convert '75%' / '2.5' / 2 / nan → float. Strips % signs."""
    if pd.isna(val):
        return default
    s = str(val).strip().replace('%', '').replace(',', '')
    try:
        return float(s)
    except:
        return default

# Parse all numeric columns we'll need
NUM_COLS = [
    'Population (mln)', 'Mobile Users (mln)', 'Mobile Penetration (%)',
    'GDP Growth (%)', 'Avg Age (Yrs)', 'Age Over 65 (%)',
    'Internet Users (%)', 'GDP per Capita (USD)',
    'Sub Base (mln)', 'Market Share (%)', 'Subscriber Growth (%)',
    '5G Penetration (%)',
]

for col in NUM_COLS:
    df[col + '_num'] = df[col].apply(to_float)

# Convenience aliases
gdp_g   = df['GDP Growth (%)_num']
avg_age = df['Avg Age (Yrs)_num']
age65   = df['Age Over 65 (%)_num']
mob_pen = df['Mobile Penetration (%)_num']
inet    = df['Internet Users (%)_num']
gdppc   = df['GDP per Capita (USD)_num']
sub_g   = df['Subscriber Growth (%)_num']
mshare  = df['Market Share (%)_num']
fiveg   = df['5G Penetration (%)_num']

# Text fields – lowercase for matching
arpu_g  = df['ARPU Growth'].fillna('').str.lower()
rev_g   = df['Revenue Growth (3 Yrs)'].fillna('').str.lower()
profit  = df['Profitability (3 Yrs)'].fillna('').str.lower()
capex   = df['Capex / Investment'].fillna('').str.lower()
fin_cmt = df['Financial Comments'].fillna('').str.lower()
reg_cmt = df['Regulation Comments'].fillna('').str.lower()
roam_cmt= df['Roaming Comments'].fillna('').str.lower()
ott     = df['OTT / International Calls'].fillna('').str.lower()
out_r   = df['Outbound Roaming Trend'].fillna('').str.lower()
in_r    = df['Inbound Roaming Trend'].fillna('').str.lower()
biz_tr  = df['Business Travellers Outbound'].fillna('').str.lower()
top_r   = df['Top Roaming Countries'].fillna('').str.lower()
prepaid = df['Prepaid / Postpaid'].fillna('').str.lower()
key_ev  = df['Key Events'].fillna('').str.lower()

print('Numeric columns parsed. Ready for rule engine.')

Numeric columns parsed. Ready for rule engine.


## 2. Helper: keyword tester

In [6]:
def kw(series, *words):
    """Return boolean Series: True if any word found in series (already lowercased)."""
    pattern = '|'.join(re.escape(w) for w in words)
    return series.str.contains(pattern, na=False)

def first_kw(series, mapping, default='Unknown'):
    """
    mapping: list of (label, [keywords])  — first match wins.
    Returns pd.Series of labels.
    """
    result = pd.Series([default] * len(series), index=series.index)
    for label, words in reversed(mapping):   # reversed so first entry wins
        mask = kw(series, *words)
        result[mask] = label
    return result

print('Helpers ready.')

Helpers ready.


## 3. Market Growth Signals

In [7]:
# ── 3.1  Population Growth ────────────────────────────────────────────────────
# GDP > 4% AND avg age < 32 → High  |  GDP 2-4% → Moderate  |  else → Low/Stable
def pop_growth(row):
    g, a = row['GDP Growth (%)_num'], row['Avg Age (Yrs)_num']
    if pd.isna(g):
        return 'Unknown'
    if g > 4 and (pd.isna(a) or a < 32):
        return 'High'
    if 2 <= g <= 4:
        return 'Moderate'
    return 'Low/Stable'

df['IA_Population_Growth'] = df.apply(pop_growth, axis=1)

# ── 3.2  Penetration ──────────────────────────────────────────────────────────
# <80% → High growth potential | 80–110% → Moderate | >110% → Saturated
def penetration(p):
    if pd.isna(p): return 'Unknown'
    if p < 80: return 'High Growth Potential'
    if p <= 110: return 'Moderate'
    return 'Saturated'

df['IA_Penetration'] = mob_pen.apply(penetration)

# ── 3.3  GDP Growth ───────────────────────────────────────────────────────────
# >5% Strong | 2-5% Moderate | <2% Weak
def gdp_class(g):
    if pd.isna(g): return 'Unknown'
    if g > 5: return 'Strong'
    if g >= 2: return 'Moderate'
    return 'Weak'

df['IA_GDP_Growth'] = gdp_g.apply(gdp_class)

# ── 3.4  Age of Population ────────────────────────────────────────────────────
# <30 → Young (high data demand) | 30-40 → Mixed | >40 → Ageing (ARPU risk)
def age_class(a):
    if pd.isna(a): return 'Unknown'
    if a < 30: return 'Young – High Data Demand'
    if a <= 40: return 'Mixed'
    return 'Ageing – ARPU Risk'

df['IA_Age_of_Population'] = avg_age.apply(age_class)

# ── 3.5  Sub Base Growth ─────────────────────────────────────────────────────
# Directly from Subscriber Growth %. Negative → Churn Risk | <3% → Stable | >=3% → Expanding
def sub_base(s):
    if pd.isna(s): return 'Unknown'
    if s < 0: return 'Churn Risk'
    if s < 3: return 'Stable'
    return 'Expanding Market'

df['IA_Sub_Base_Growth'] = sub_g.apply(sub_base)

print('Market growth signals done.')
df[['IA_Population_Growth','IA_Penetration','IA_GDP_Growth','IA_Age_of_Population','IA_Sub_Base_Growth']].value_counts().head()

Market growth signals done.


IA_Population_Growth  IA_Penetration  IA_GDP_Growth  IA_Age_of_Population  IA_Sub_Base_Growth
Low/Stable            Saturated       Weak           Ageing – ARPU Risk    Unknown               87
Moderate              Saturated       Moderate       Ageing – ARPU Risk    Unknown               56
                                                     Mixed                 Unknown               31
                                                                           Stable                26
Unknown               Saturated       Unknown        Ageing – ARPU Risk    Stable                21
Name: count, dtype: int64

## 4. ARPU & Revenue Signals

In [8]:
# ── 4.1  ARPU Impact ──────────────────────────────────────────────────────────
# Rule: ARPU Growth text + Prepaid share + 5G penetration
def arpu_impact(row):
    a   = str(row['ARPU Growth']).lower()
    pre = str(row['Prepaid / Postpaid']).lower()
    fg  = row['5G Penetration (%)_num']

    # 5G uplift signal
    if not pd.isna(fg) and fg > 50:
        return 'Potential Upsell – 5G Uplift'

    # Prepaid dominant → slow
    pre_pct = re.search(r'(\d+)%\s*pre', pre)
    if pre_pct and int(pre_pct.group(1)) >= 60:
        if any(w in a for w in ['flat', 'declining', 'slow', 'marginal']):
            return 'Slow/Declining'

    if any(w in a for w in ['increasing', 'growing', 'strong', 'positive']):
        return 'Growing'
    if any(w in a for w in ['marginal up', 'marginal']):
        return 'Marginal Up – Slow Growth'
    if any(w in a for w in ['flat', 'stable', 'neutral']):
        return 'Slow'
    if any(w in a for w in ['declining', 'down', 'negative']):
        return 'Declining'
    return 'Slow'

df['IA_ARPU_Impact'] = df.apply(arpu_impact, axis=1)

# ── 4.2  Outbound Impact ─────────────────────────────────────────────────────
df['IA_Outbound_Impact'] = first_kw(out_r, [
    ('High Positive',   ['growing 15', 'strong outbound', 'high per capita', 'increasing']),
    ('Moderate Positive', ['moderate outbound', 'moderate', 'regional travel']),
    ('Negative / Declining', ['declining', 'very limited', 'limited', 'n/a', 'not a retail']),
    ('Low',             ['low', 'minimal']),
], default='Moderate')

# ── 4.3  Biz Traveller Impact ─────────────────────────────────────────────────
def biz_impact(row):
    biz = str(row['Business Travellers Outbound']).lower()
    gdp = row['GDP per Capita (USD)_num']
    if 'n/a' in biz or 'not a retail' in biz:
        return 'N/A'
    high_gdp = (not pd.isna(gdp)) and gdp > 20000
    if any(w in biz for w in ['high', 'strong', 'oil sector']):
        return 'Strong Impact' if high_gdp else 'Moderate Impact'
    if any(w in biz for w in ['moderate', 'medium']):
        return 'Moderate Impact'
    return 'Low Impact'

df['IA_Biz_Traveller_Impact'] = df.apply(biz_impact, axis=1)

# ── 4.4  Inbound Impact ──────────────────────────────────────────────────────
df['IA_Inbound_Impact'] = first_kw(in_r, [
    ('Major – Tourism/Business Hub', ['major inbound', 'tourism/business', 'hajj', 'religious']),
    ('Growing – Religious/Events',   ['growing', 'religious tourism']),
    ('Moderate – Regional Visitors', ['moderate inbound', 'moderate', 'regional']),
    ('Low',                          ['low', 'minimal', 'limited']),
    ('Negative – Conflict',          ['conflict', 'none – conflict', 'impacted by conflict']),
    ('N/A',                          ['n/a', 'not a retail']),
], default='Moderate – Regional Visitors')

# ── 4.5  Immigration Impact ──────────────────────────────────────────────────
def immigration(row):
    rc = str(row['Roaming Comments']).lower()
    ir = str(row['Inbound Roaming Trend']).lower()
    if 'n/a' in ir or 'not a retail' in ir:
        return 'N/A'
    if any(w in rc for w in ['diaspora', 'migrant', 'expat', 'labour']):
        return 'Moderate to High – Diaspora/Migrant'
    if any(w in rc for w in ['tourism', 'tourist', 'visitor']):
        return 'Low – Tourism Only'
    return 'Low'

df['IA_Immigration_Impact'] = df.apply(immigration, axis=1)

# ── 4.6  International Calls ─────────────────────────────────────────────────
def intl_calls(o):
    if any(w in o for w in ['restricted', 'partially restricted', 'blocked']):
        return 'IDD Declining – OTT Restrictions Persist'
    if any(w in o for w in ['whatsapp', 'viber', 'ott dominant', 'ott calls']):
        return 'IDD Declining – OTT Dominant'
    if any(w in o for w in ['all voip', 'all ott', 'fully open']):
        return 'Traditional Calls Declining – Full OTT'
    if any(w in o for w in ['facebook', 'messenger']):
        return 'OTT Impact – Moderate'
    return 'Traditional Calls Persist'

df['IA_International_Calls'] = ott.apply(intl_calls)

# ── 4.7  Apps ────────────────────────────────────────────────────────────────
def apps(row):
    o   = str(row['OTT / International Calls']).lower()
    iu  = row['Internet Users (%)_num']
    if pd.isna(iu): iu = 50
    high_inet = iu > 70
    if high_inet and any(w in o for w in ['whatsapp', 'ott dominant', 'viber', 'all voip', 'all ott']):
        return 'High App Substitution Risk'
    if high_inet:
        return 'Moderate App Impact'
    return 'Low App Substitution Risk'

df['IA_Apps'] = df.apply(apps, axis=1)

print('ARPU & Revenue signals done.')

ARPU & Revenue signals done.


## 5. Revenue & Profitability Signals

In [9]:
# ── 5.1  Rev Growth ──────────────────────────────────────────────────────────
df['IA_Rev_Growth'] = first_kw(rev_g, [
    ('Growing',          ['increasing', 'growing', 'strong growth']),
    ('Marginal Up',      ['marginal incr', 'marginal growth', 'marginal up']),
    ('Flat',             ['flat', 'stable revenue', 'stable', 'moderate']),
    ('Declined',         ['declining', 'declined', 'decreasing', 'negative']),
], default='Flat')

# ── 5.2  Profitability ───────────────────────────────────────────────────────
df['IA_Profitability'] = first_kw(profit, [
    ('Up',   ['up', 'ebitda positive', 'good ebitda', 'improving', 'strong', 'high']),
    ('Flat', ['flat', 'stable', 'moderate', 'neutral']),
    ('Down', ['down', 'low', 'impacting margins', 'declining', 'negative', 'loss']),
], default='Flat')

# ── 5.3  Future Investment Cash Flow ─────────────────────────────────────────
def future_cf(row):
    c  = str(row['Capex / Investment']).lower()
    fc = str(row['Financial Comments']).lower()
    if any(w in c for w in ['5g rollout', '5g expansion', '5g pilot', 'sar', 'usd', 'b capex', 'billion']):
        return 'Up – Cash Committed for 5G/Infra'
    if any(w in c for w in ['planned upgrade', '4g upgrade', 'expansion', 'upgrade']):
        return 'Up – Planned Upgrade'
    if any(w in c for w in ['moderate', 'medium']):
        return 'Neutral – Steady Investment'
    if any(w in c for w in ['low', 'minimal', 'none']):
        return 'Low – Minimal Capex'
    return 'Neutral'

df['IA_Future_Investment_CF'] = df.apply(future_cf, axis=1)

print('Revenue & profitability signals done.')

Revenue & profitability signals done.


## 6. Need for Managed Services Signals

In [10]:
# ── 6.1  Need for MS – Financial ─────────────────────────────────────────────
# Score: low GDP/cap + declining rev + flat/down profitability → High MS need
def ms_financial(row):
    score = 0
    gdp   = row['GDP per Capita (USD)_num']
    rg    = str(row['Revenue Growth (3 Yrs)']).lower()
    pr    = str(row['Profitability (3 Yrs)']).lower()

    if not pd.isna(gdp) and gdp < 5000:
        score += 2
    elif not pd.isna(gdp) and gdp < 15000:
        score += 1

    if any(w in rg for w in ['declining', 'declined', 'flat', 'stable']):
        score += 1
    if any(w in pr for w in ['down', 'low', 'impacting', 'flat']):
        score += 1

    if score >= 3: return 'High MS Need – Weak Financials'
    if score == 2: return 'Medium MS Need'
    return 'Low MS Need – Strong Financials'

df['IA_Need_MS_Financial'] = df.apply(ms_financial, axis=1)

# ── 6.2  Need for MS – Technical ─────────────────────────────────────────────
# 5G < 10% + low capex → High | 5G 10–40% → Medium | 5G > 40% → Low
def ms_technical(row):
    fg = row['5G Penetration (%)_num']
    c  = str(row['Capex / Investment']).lower()
    low_capex = any(w in c for w in ['low', 'minimal', 'none'])
    if pd.isna(fg): fg = 0
    if fg < 10 and low_capex:
        return 'High MS Need – Low 5G + Low Capex'
    if fg < 10:
        return 'High MS Need – Low 5G'
    if fg <= 40:
        return 'Medium MS Need'
    return 'Low MS Need – Advanced 5G'

df['IA_Need_MS_Technical'] = df.apply(ms_technical, axis=1)

# ── 6.3  Need for MS – Package Deal ──────────────────────────────────────────
# Regulatory pressure + margin squeeze + capex obligation → High package deal potential
def ms_package(row):
    rc  = str(row['Regulation Comments']).lower()
    fc  = str(row['Financial Comments']).lower()
    c   = str(row['Capex / Investment']).lower()
    pr  = str(row['Profitability (3 Yrs)']).lower()
    score = 0
    if any(w in rc for w in ['obligation', 'mandate', 'required', 'must', 'penalty']):
        score += 1
    if any(w in pr for w in ['down', 'low', 'impacting', 'flat', 'margin']):
        score += 1
    if any(w in c for w in ['5g rollout', 'upgrade', 'expansion', 'planned']):
        score += 1

    if score >= 2: return 'High Package Deal Potential'
    if score == 1: return 'Medium Package Deal Potential'
    return 'Low Package Deal Potential'

df['IA_Need_MS_Package'] = df.apply(ms_package, axis=1)

print('MS need signals done.')

MS need signals done.


## 7. Investment & Business Model Signals

In [11]:
# ── 7.1  Future Investment UC ─────────────────────────────────────────────────
# High 5G + growing rev + active capex → Good investment
# Low 5G + stagnant rev → Asset rotation or May not be too high
def future_uc(row):
    fg  = row['5G Penetration (%)_num']
    rg  = str(row['Revenue Growth (3 Yrs)']).lower()
    c   = str(row['Capex / Investment']).lower()
    if pd.isna(fg): fg = 0

    growing_rev = any(w in rg for w in ['growing', 'increasing', 'marginal incr'])
    active_capex = any(w in c for w in ['5g', 'rollout', 'expansion', 'upgrade', 'planned'])

    if fg > 40 and growing_rev and active_capex:
        return 'Good Investment – Strong 5G + Revenue Growth'
    if fg > 40 and growing_rev:
        return 'Good Investment'
    if fg < 20 and not growing_rev:
        return 'Asset Rotation Candidate'
    if fg < 10:
        return 'May Not Be Too High – Low Maturity'
    return 'Moderate Investment Opportunity'

df['IA_Future_Investment_UC'] = df.apply(future_uc, axis=1)

# ── 7.2  Financials (Composite Summary) ───────────────────────────────────────
def financials_summary(row):
    rg  = str(row['Revenue Growth (3 Yrs)']).lower()
    pr  = str(row['Profitability (3 Yrs)']).lower()
    gdp = row['GDP per Capita (USD)_num']

    rev_label  = 'Growing' if any(w in rg for w in ['growing','increasing']) else \
                 'Marginal' if 'marginal' in rg else \
                 'Declining' if any(w in rg for w in ['declin','declined']) else 'Stable'

    prof_label = 'Profitable' if any(w in pr for w in ['up','good ebitda','improving','strong']) else \
                 'Under Pressure' if any(w in pr for w in ['down','low','impacting','loss']) else 'Stable'

    wealth = '' if pd.isna(gdp) else \
             ' | Wealthy Country' if gdp > 25000 else \
             ' | Mid-Income Country' if gdp > 8000 else \
             ' | Low-Income Country'

    return f'Rev: {rev_label} | Profit: {prof_label}{wealth}'

df['IA_Financials'] = df.apply(financials_summary, axis=1)

# ── 7.3  Scope of MS ─────────────────────────────────────────────────────────
# SaaS model likely when: digital maturity + revenue growth
# MS likely when: capex obligations + technical gaps
def scope_ms(row):
    fc  = str(row['Financial Comments']).lower()
    fg  = row['5G Penetration (%)_num']
    pr  = str(row['Profitability (3 Yrs)']).lower()
    rg  = str(row['Revenue Growth (3 Yrs)']).lower()
    c   = str(row['Capex / Investment']).lower()
    if pd.isna(fg): fg = 0

    saas_signals   = fg > 40 and any(w in rg for w in ['growing', 'increasing', 'marginal incr'])
    ms_signals     = any(w in c for w in ['upgrade', '5g', 'rollout', 'expansion']) and \
                     any(w in pr for w in ['down', 'flat', 'low', 'impacting'])

    if saas_signals and ms_signals:
        return 'Good Possibility – SaaS Model + MS'
    if saas_signals:
        return 'SaaS Model Likely – Digital Maturity'
    if ms_signals:
        return 'Managed Services Likely – Capex + Tech Gaps'
    return 'Moderate – Evaluate Case by Case'

df['IA_Scope_of_MS'] = df.apply(scope_ms, axis=1)

print('Investment & business model signals done.')

Investment & business model signals done.


## 8. Roaming & Competitive Signals

In [12]:
# ── 8.1  International In-Roamers ─────────────────────────────────────────────
def in_roamers(row):
    ir  = str(row['Inbound Roaming Trend']).lower()
    tr  = str(row['Top Roaming Countries']).lower()
    if 'n/a' in ir or 'not a retail' in ir:
        return 'N/A'
    country_count = len([x for x in tr.split(',') if x.strip()]) if tr else 0
    if country_count >= 4 and any(w in ir for w in ['growing', 'major', 'high', 'religious', 'hajj']):
        return 'High – Diverse In-Roamer Base'
    if any(w in ir for w in ['moderate', 'regional', 'tourism']):
        return 'Moderate In-Roamer Base'
    if any(w in ir for w in ['conflict', 'none', 'low']):
        return 'Low – Conflict/Limited Tourism'
    return 'Moderate In-Roamer Base'

df['IA_Intl_In_Roamers'] = df.apply(in_roamers, axis=1)

# ── 8.2  Outroamers ──────────────────────────────────────────────────────────
def outroamers(row):
    or_ = str(row['Outbound Roaming Trend']).lower()
    tr  = str(row['Top Roaming Countries']).lower()
    gdp = row['GDP per Capita (USD)_num']
    if 'n/a' in or_ or 'not a retail' in or_:
        return 'N/A'
    wealthy = (not pd.isna(gdp)) and gdp > 20000
    if any(w in or_ for w in ['growing', 'strong', 'high per capita', 'increasing']) and wealthy:
        return 'Active Outroamer Base – Wealthy'
    if any(w in or_ for w in ['moderate', 'regional']):
        return 'Moderate Outroamer Activity'
    if any(w in or_ for w in ['limited', 'very limited', 'low']):
        return 'Low Outroamer Activity'
    return 'Moderate Outroamer Activity'

df['IA_Outroamers'] = df.apply(outroamers, axis=1)

# ── 8.3  BU Gaps ─────────────────────────────────────────────────────────────
# Low 5G + growing rev → Tech gap | High churn + low capex → Retention gap
def bu_gaps(row):
    fg  = row['5G Penetration (%)_num']
    rg  = str(row['Revenue Growth (3 Yrs)']).lower()
    sg  = row['Subscriber Growth (%)_num']
    c   = str(row['Capex / Investment']).lower()
    if pd.isna(fg): fg = 0
    gaps = []
    if fg < 20 and any(w in rg for w in ['growing', 'increasing', 'marginal incr']):
        gaps.append('Technology Gap')
    if (not pd.isna(sg)) and sg < 0 and any(w in c for w in ['low', 'minimal']):
        gaps.append('Retention Gap')
    if fg < 10:
        gaps.append('5G Coverage Gap')
    return ' | '.join(gaps) if gaps else 'No Critical Gap Identified'

df['IA_BU_Gaps'] = df.apply(bu_gaps, axis=1)

# ── 8.4  Recommended Solutions ────────────────────────────────────────────────
def rec_solutions(row):
    gaps = row['IA_BU_Gaps']
    fg   = row['5G Penetration (%)_num']
    or_  = str(row['Outbound Roaming Trend']).lower()
    if pd.isna(fg): fg = 0
    sols = []
    if 'Technology Gap' in gaps or fg < 20:
        sols.append('Infrastructure MS')
    if 'Retention Gap' in gaps:
        sols.append('SaaS / CRM Solutions')
    if '5G Coverage Gap' in gaps:
        sols.append('Roaming Hub + 5G Rollout Support')
    if any(w in or_ for w in ['growing', 'strong', 'high']):
        sols.append('Roaming Hub')
    return ' | '.join(sols) if sols else 'SaaS / Digital Transformation'

df['IA_Recommended_Solutions'] = df.apply(rec_solutions, axis=1)

# ── 8.5  Recommended Biz Model ───────────────────────────────────────────────
# Market leader + high profitability → Partnership
# Struggling → MS/Managed Services | Mid-tier → SaaS
def rec_biz_model(row):
    ms  = row['Market Share (%)_num']
    pr  = str(row['Profitability (3 Yrs)']).lower()
    fc  = str(row['Financial Comments']).lower()
    high_profit = any(w in pr for w in ['up','good ebitda','strong','improving'])
    low_profit  = any(w in pr for w in ['down','low','impacting','loss','flat'])
    if (not pd.isna(ms)) and ms > 40 and high_profit:
        return 'Partnership – Market Leader'
    if low_profit or ((not pd.isna(ms)) and ms < 20):
        return 'Managed Services – Struggling Operator'
    return 'SaaS – Mid-Tier Operator'

df['IA_Recommended_Biz_Model'] = df.apply(rec_biz_model, axis=1)

# ── 8.6  Satisfaction of Competitive Product ─────────────────────────────────
def satisfaction(row):
    sg = row['Subscriber Growth (%)_num']
    ms = row['Market Share (%)_num']
    if pd.isna(sg) or pd.isna(ms):
        return 'Unknown'
    if sg < 0 and ms < 25:
        return 'Low Satisfaction – Declining Sub + Shrinking Share'
    if sg > 2 and ms > 35:
        return 'High Satisfaction – Growing Share'
    return 'Medium Satisfaction – Stable'

df['IA_Satisfaction_Competitive'] = df.apply(satisfaction, axis=1)

# ── 8.7  Cost to Replace Competition ─────────────────────────────────────────
def cost_replace(row):
    fg  = row['5G Penetration (%)_num']
    c   = str(row['Capex / Investment']).lower()
    gdp = row['GDP per Capita (USD)_num']
    if pd.isna(fg): fg = 0
    high_capex  = any(w in c for w in ['5g rollout', '5g expansion', 'sar', 'usd', 'billion', 'committed'])
    if fg > 50 and high_capex:
        return 'High Switching Cost – Committed 5G + High 5G Maturity'
    if fg < 20 and not high_capex:
        return 'Low Cost to Displace – Low 5G + Low Capex'
    return 'Medium Switching Cost'

df['IA_Cost_Replace_Competition'] = df.apply(cost_replace, axis=1)

# ── 8.8  Approach to Replace – Lic or MS ─────────────────────────────────────
def approach_replace(row):
    fc  = str(row['Financial Comments']).lower()
    pr  = str(row['Profitability (3 Yrs)']).lower()
    c   = str(row['Capex / Investment']).lower()
    gdp = row['GDP per Capita (USD)_num']
    has_budget = (not pd.isna(gdp)) and gdp > 15000
    has_tech_gap = any(w in c for w in ['upgrade', '5g', 'expansion'])
    cash_flow_issue = any(w in pr for w in ['down', 'low', 'impacting', 'flat'])

    if has_budget and has_tech_gap:
        return 'License Deal – Budget + Tech Gap'
    if cash_flow_issue:
        return 'Managed Services – Cash Flow Issues + Operational Gap'
    return 'Evaluate: License or MS Based on Negotiation'

df['IA_Approach_Replace'] = df.apply(approach_replace, axis=1)

print('Roaming & competitive signals done.')

Roaming & competitive signals done.


## 9. Summary & Validation

In [13]:
IA_COLS = [
    'IA_Population_Growth', 'IA_Penetration', 'IA_GDP_Growth',
    'IA_Age_of_Population', 'IA_Sub_Base_Growth',
    'IA_ARPU_Impact', 'IA_Outbound_Impact', 'IA_Biz_Traveller_Impact',
    'IA_Inbound_Impact', 'IA_Immigration_Impact', 'IA_International_Calls', 'IA_Apps',
    'IA_Rev_Growth', 'IA_Profitability', 'IA_Future_Investment_CF',
    'IA_Need_MS_Financial', 'IA_Need_MS_Technical', 'IA_Need_MS_Package',
    'IA_Future_Investment_UC', 'IA_Financials', 'IA_Scope_of_MS',
    'IA_Intl_In_Roamers', 'IA_Outroamers',
    'IA_BU_Gaps', 'IA_Recommended_Solutions', 'IA_Recommended_Biz_Model',
    'IA_Satisfaction_Competitive', 'IA_Cost_Replace_Competition', 'IA_Approach_Replace'
]

# Check null / Unknown counts
print('=== Prediction Coverage ===')
for col in IA_COLS:
    total = len(df)
    unknowns = (df[col] == 'Unknown').sum() + (df[col] == 'N/A').sum()
    pct = round(100 * (total - unknowns) / total, 1)
    print(f'  {col.replace("IA_",""):35s}  coverage: {pct}%')

print(f'\nTotal operators: {len(df)}')
print(f'Total IA columns predicted: {len(IA_COLS)}')

=== Prediction Coverage ===
  Population_Growth                    coverage: 84.7%
  Penetration                          coverage: 100.0%
  GDP_Growth                           coverage: 84.7%
  Age_of_Population                    coverage: 100.0%
  Sub_Base_Growth                      coverage: 48.2%
  ARPU_Impact                          coverage: 100.0%
  Outbound_Impact                      coverage: 100.0%
  Biz_Traveller_Impact                 coverage: 77.0%
  Inbound_Impact                       coverage: 77.0%
  Immigration_Impact                   coverage: 77.0%
  International_Calls                  coverage: 100.0%
  Apps                                 coverage: 100.0%
  Rev_Growth                           coverage: 100.0%
  Profitability                        coverage: 100.0%
  Future_Investment_CF                 coverage: 100.0%
  Need_MS_Financial                    coverage: 100.0%
  Need_MS_Technical                    coverage: 100.0%
  Need_MS_Package         

## 10. Sample Preview

In [14]:
# Preview first 5 operators with all IA predictions
DISPLAY_COLS = ['Region', 'Country', 'Operator Name'] + IA_COLS
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 45)
df[DISPLAY_COLS].head(5)

,Region,Country,Operator Name,IA_Population_Growth,IA_Penetration,IA_GDP_Growth,IA_Age_of_Population,IA_Sub_Base_Growth,IA_ARPU_Impact,IA_Outbound_Impact,IA_Biz_Traveller_Impact,IA_Inbound_Impact,IA_Immigration_Impact,IA_International_Calls,IA_Apps,IA_Rev_Growth,IA_Profitability,IA_Future_Investment_CF,IA_Need_MS_Financial,IA_Need_MS_Technical,IA_Need_MS_Package,IA_Future_Investment_UC,IA_Financials,IA_Scope_of_MS,IA_Intl_In_Roamers,IA_Outroamers,IA_BU_Gaps,IA_Recommended_Solutions,IA_Recommended_Biz_Model,IA_Satisfaction_Competitive,IA_Cost_Replace_Competition,IA_Approach_Replace
0,MECA,Saudi Arabia,STC,Moderate,Saturated,Moderate,Young – High Data Demand,Stable,Potential Upsell – 5G Uplift,High Positive,Strong Impact,Major – Tourism/Business Hub,Low,IDD Declining – OTT Restrictions Persist,High App Substitution Risk,Growing,Up,Up – Cash Committed for 5G/Infra,Low MS Need – Strong Financials,Low MS Need – Advanced 5G,Low Package Deal Potential,Good Investment – Strong 5G + Revenue Growth,Rev: Growing | Profit: Profitable | Wealt...,SaaS Model Likely – Digital Maturity,High – Diverse In-Roamer Base,Active Outroamer Base – Wealthy,No Critical Gap Identified,Roaming Hub,Partnership – Market Leader,Medium Satisfaction – Stable,High Switching Cost – Committed 5G + High...,License Deal – Budget + Tech Gap
1,MECA,Saudi Arabia,Mobily,Moderate,Saturated,Moderate,Young – High Data Demand,Stable,Potential Upsell – 5G Uplift,High Positive,Strong Impact,Major – Tourism/Business Hub,Low,IDD Declining – OTT Restrictions Persist,High App Substitution Risk,Marginal Up,Flat,Up – Planned Upgrade,Low MS Need – Strong Financials,Low MS Need – Advanced 5G,High Package Deal Potential,Good Investment – Strong 5G + Revenue Growth,Rev: Marginal | Profit: Stable | Wealthy ...,Good Possibility – SaaS Model + MS,High – Diverse In-Roamer Base,Active Outroamer Base – Wealthy,No Critical Gap Identified,Roaming Hub,Managed Services – Struggling Operator,Medium Satisfaction – Stable,Medium Switching Cost,License Deal – Budget + Tech Gap
2,MECA,Saudi Arabia,Zain KSA,Moderate,Saturated,Moderate,Young – High Data Demand,Stable,Slow/Declining,High Positive,Strong Impact,Major – Tourism/Business Hub,Low,IDD Declining – OTT Restrictions Persist,High App Substitution Risk,Growing,Up,Neutral,Low MS Need – Strong Financials,Low MS Need – Advanced 5G,Low Package Deal Potential,Good Investment – Strong 5G + Revenue Growth,Rev: Growing | Profit: Profitable | Wealt...,SaaS Model Likely – Digital Maturity,High – Diverse In-Roamer Base,Active Outroamer Base – Wealthy,No Critical Gap Identified,Roaming Hub,SaaS – Mid-Tier Operator,Medium Satisfaction – Stable,Medium Switching Cost,License Deal – Budget + Tech Gap
3,MECA,UAE,e& (Etisalat),Moderate,Saturated,Moderate,Mixed,Stable,Potential Upsell – 5G Uplift,Moderate,Strong Impact,Moderate – Regional Visitors,Moderate to High – Diaspora/Migrant,IDD Declining – OTT Restrictions Persist,Moderate App Impact,Growing,Up,Neutral,Low MS Need – Strong Financials,Low MS Need – Advanced 5G,Low Package Deal Potential,Good Investment,Rev: Growing | Profit: Profitable | Wealt...,SaaS Model Likely – Digital Maturity,High – Diverse In-Roamer Base,Moderate Outroamer Activity,No Critical Gap Identified,Roaming Hub,Partnership – Market Leader,Medium Satisfaction – Stable,Medium Switching Cost,Evaluate: License or MS Based on Negotiation
4,MECA,UAE,du,Moderate,Saturated,Moderate,Mixed,Stable,Potential Upsell – 5G Uplift,Moderate,Strong Impact,Moderate – Regional Visitors,Moderate to High – Diaspora/Migrant,IDD Declining – OTT Restrictions Persist,Moderate App Impact,Growing,Up,Up – Cash Committed for 5G/Infra,Low MS Need – Strong Financials,Low MS Need – Advanced 5G,Medium Package Deal Potential,Good Investment – Strong 5G + Revenue Growth,Rev: Growing | Profit: Profitable | Wealt...,SaaS Model Likely – Digital Maturity,High – Diverse In-Roamer Base,Moderate Outroamer Activity,No Critical Gap Identified,Roaming Hub,SaaS – Mid-Tier Operator,Mediu

## 11. Export to Excel (original data + Impact Analysis sheet)

In [15]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import copy

OUTPUT_FILE = 'Global_Telecom_MNO_ImpactAnalysis.xlsx'

# ── 11.1 Write main dataframe back with IA columns appended ──────────────────
BASE_COLS = [
    'Region', 'Sub-Region', 'Country', 'Operator Name',
    'GDP Growth (%)', 'Mobile Penetration (%)', 'Avg Age (Yrs)',
    'Subscriber Growth (%)', '5G Penetration (%)',
    'ARPU Growth', 'Revenue Growth (3 Yrs)', 'Profitability (3 Yrs)',
    'Capex / Investment', 'Outbound Roaming Trend', 'Inbound Roaming Trend',
    'Business Travellers Outbound', 'OTT / International Calls', 'Market Share (%)',
]

REGIONS = ['MECA', 'Europe', 'APAC', 'LATAM']

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    # Combined sheet
    df[BASE_COLS + IA_COLS].to_excel(writer, sheet_name='Combined Impact Analysis', index=False)

    # Per-region sheets
    for region in REGIONS:
        region_df = df[df['Region'].str.contains(region, na=False)]
        region_df[BASE_COLS + IA_COLS].to_excel(writer, sheet_name=f'{region} Impact Analysis', index=False)

print(f'Written {OUTPUT_FILE}')

Written Global_Telecom_MNO_ImpactAnalysis.xlsx


In [16]:
# ── 11.2 Style the workbook ─────────────────────────────────────────────────
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

HEADER_FILL    = PatternFill('solid', start_color='1F3864')  # dark navy
IA_FILL        = PatternFill('solid', start_color='C6EFCE')  # green tint for IA cols
IA_HEADER_FILL = PatternFill('solid', start_color='375623')  # dark green for IA headers
ALT_FILL       = PatternFill('solid', start_color='EBF3FB')  # light blue alternate row
THIN_BORDER    = Border(
    left   = Side(style='thin', color='CCCCCC'),
    right  = Side(style='thin', color='CCCCCC'),
    top    = Side(style='thin', color='CCCCCC'),
    bottom = Side(style='thin', color='CCCCCC'),
)

wb = load_workbook(OUTPUT_FILE)

for ws_name in wb.sheetnames:
    ws = wb[ws_name]
    max_col = ws.max_column
    max_row = ws.max_row

    # Find which columns are IA columns
    ia_col_idxs = set()
    for col_idx in range(1, max_col + 1):
        h = ws.cell(row=1, column=col_idx).value or ''
        if str(h).startswith('IA_'):
            ia_col_idxs.add(col_idx)

    # Style headers
    for col_idx in range(1, max_col + 1):
        cell = ws.cell(row=1, column=col_idx)
        if col_idx in ia_col_idxs:
            cell.fill = IA_HEADER_FILL
            cell.font = Font(bold=True, color='FFFFFF', size=9, name='Arial')
        else:
            cell.fill = HEADER_FILL
            cell.font = Font(bold=True, color='FFFFFF', size=9, name='Arial')
        cell.alignment = Alignment(wrap_text=True, horizontal='center', vertical='center')
        cell.border = THIN_BORDER

    # Style data rows
    for row_idx in range(2, max_row + 1):
        is_alt = (row_idx % 2 == 0)
        for col_idx in range(1, max_col + 1):
            cell = ws.cell(row=row_idx, column=col_idx)
            cell.border = THIN_BORDER
            cell.font   = Font(size=8, name='Arial')
            cell.alignment = Alignment(wrap_text=True, vertical='top')
            if col_idx in ia_col_idxs:
                cell.fill = IA_FILL
            elif is_alt:
                cell.fill = ALT_FILL

    # Column widths
    for col_idx in range(1, max_col + 1):
        header = str(ws.cell(row=1, column=col_idx).value or '')
        if col_idx in ia_col_idxs:
            ws.column_dimensions[get_column_letter(col_idx)].width = 30
        elif any(x in header for x in ['Comment', 'Regulation', 'Events']):
            ws.column_dimensions[get_column_letter(col_idx)].width = 20
        else:
            ws.column_dimensions[get_column_letter(col_idx)].width = 18

    # Freeze header row
    ws.freeze_panes = 'A2'

    # Auto-filter
    ws.auto_filter.ref = ws.dimensions

    # Row height for header
    ws.row_dimensions[1].height = 40

wb.save(OUTPUT_FILE)
print(f'Styled workbook saved: {OUTPUT_FILE}')

Styled workbook saved: Global_Telecom_MNO_ImpactAnalysis.xlsx


## 12. Region-level Aggregate Summary

In [17]:
summary_rows = []
for region in ['MECA', 'Europe', 'APAC', 'LATAM']:
    rdf = df[df['Region'].str.contains(region, na=False)]
    row = {
        'Region': region,
        'Operators': len(rdf),
        'GDP Growth (avg)': round(rdf['GDP Growth (%)_num'].mean(), 2),
        '5G Pen (avg %)': round(rdf['5G Penetration (%)_num'].mean(), 1),
        'Sub Growth (avg %)': round(rdf['Subscriber Growth (%)_num'].mean(), 2),
        'Top ARPU Impact': rdf['IA_ARPU_Impact'].value_counts().idxmax(),
        'Top Rev Growth': rdf['IA_Rev_Growth'].value_counts().idxmax(),
        'Top Biz Model': rdf['IA_Recommended_Biz_Model'].value_counts().idxmax(),
        'Top MS Need (Fin)': rdf['IA_Need_MS_Financial'].value_counts().idxmax(),
        'Top Scope of MS': rdf['IA_Scope_of_MS'].value_counts().idxmax(),
    }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

Region  Operators  GDP Growth (avg)  5G Pen (avg %)  Sub Growth (avg %) Top ARPU Impact Top Rev Growth                          Top Biz Model               Top MS Need (Fin)                  Top Scope of MS
  MECA        142              3.64            24.2                0.71  Slow/Declining           Flat Managed Services – Struggling Operator Low MS Need – Strong Financials Moderate – Evaluate Case by Case
Europe        204              1.75             NaN                0.73            Slow           Flat               SaaS – Mid-Tier Operator Low MS Need – Strong Financials Moderate – Evaluate Case by Case
  APAC         81              2.50            28.2                1.34            Slow           Flat               SaaS – Mid-Tier Operator Low MS Need – Strong Financials Moderate – Evaluate Case by Case
 LATAM         90              2.71             NaN                0.99            Slow           Flat               SaaS – Mid-Tier Operator Low MS Need – Strong Financial

In [18]:
# Append summary to workbook
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment

wb = load_workbook(OUTPUT_FILE)
ws = wb.create_sheet('Regional Summary')

# Write summary_df
for c_idx, col_name in enumerate(summary_df.columns, 1):
    cell = ws.cell(row=1, column=c_idx, value=col_name)
    cell.font  = Font(bold=True, color='FFFFFF', name='Arial', size=10)
    cell.fill  = PatternFill('solid', start_color='1F3864')
    cell.alignment = Alignment(wrap_text=True, horizontal='center')

for r_idx, row in summary_df.iterrows():
    for c_idx, val in enumerate(row, 1):
        cell = ws.cell(row=r_idx + 2, column=c_idx, value=val)
        cell.font = Font(name='Arial', size=9)
        cell.alignment = Alignment(wrap_text=True)

for col_idx in range(1, len(summary_df.columns) + 1):
    ws.column_dimensions[get_column_letter(col_idx)].width = 28

wb.save(OUTPUT_FILE)
print(f'Regional Summary sheet added → {OUTPUT_FILE}')

Regional Summary sheet added → Global_Telecom_MNO_ImpactAnalysis.xlsx


## ✅ Done!

The output file `Global_Telecom_MNO_ImpactAnalysis.xlsx` contains:

| Sheet | Contents |
|---|---|
| Combined Impact Analysis | All 517 operators + 29 IA predictions |
| MECA Impact Analysis | 105 MECA operators + IA predictions |
| Europe Impact Analysis | 204 European operators + IA predictions |
| APAC Impact Analysis | 81 APAC operators + IA predictions |
| LATAM Impact Analysis | 90 LATAM operators + IA predictions |
| Regional Summary | Aggregate stats by region |

### Columns predicted (29 total)
- **Market Growth (5):** Population Growth, Penetration, GDP Growth, Age of Population, Sub Base Growth
- **ARPU / Roaming (7):** ARPU Impact, Outbound, Biz Traveller, Inbound, Immigration, Intl Calls, Apps
- **Financial (3):** Rev Growth, Profitability, Future Investment Cash Flow
- **MS Need (3):** Need MS Financial, Need MS Technical, Need MS Package
- **Investment & Biz (3):** Future Investment UC, Financials, Scope of MS
- **Roaming (2):** Intl In-Roamers, Outroamers
- **Competitive (6):** BU Gaps, Recommended Solutions, Recommended Biz Model, Satisfaction, Cost to Replace, Approach to Replace